In [1]:
import pandas as pd
import sys

sys.path.append('..')
from src.features.augmentation import (
    realistic_positive_augmentation,
)

In [2]:
# ============================================================
# LOAD TRAIN PAIRS
# ============================================================

train_pairs = pd.read_parquet(
    "../data/processed/train_pairs.parquet"
)

print(f"Train pairs: {len(train_pairs):,}")

Train pairs: 120,894


In [3]:
# ============================================================
# SAMPLE POSITIVE NAMES
# ============================================================

positive = train_pairs[
    train_pairs["label"] == 1
].copy()

sample = positive.sample(
    50,
    random_state=42,
)

examples = []

for _, row in sample.iterrows():
    original = row["name_latin_1"]

    augmented = realistic_positive_augmentation(original)

    examples.append(
        {
            "original": original,
            "augmented": augmented,
        }
    )

aug_examples = pd.DataFrame(examples)

display(aug_examples)

,original,augmented
0,nugmanov dorobiddin meliknorovich,nugmanov dorobiddin eliknorovich
1,samadov iskandar davlyatovich,samadov iskandar davlyatovich
2,saidov tagoi murod satorovich,tagoi saidov murod satorovich
3,abduvalieva barno muhamedchonovna,abduvalieva barno
4,davlatov shukrihudo tuhtamishovich,shukrihudo davlatov tuhtamishovich
5,kie muddinov suҳrob amridinovich,kie muddinov suҳrob amridinovich
6,anvarova barfi yusupovna,anvarova barfi yusupovna
7,muhshulova sanifa rizoevna,muhshulova sanifa
8,bobochonov abdumanon abdukodirovich,bobochonov abdumanon abdukodirovich
9,saidov safarhon karaevich,saidov safarhon karaevich


In [4]:
# ============================================================
# CHECK TOO AGGRESSIVE AUGMENTATIONS
# ============================================================

aug_examples["original_tokens"] = (
    aug_examples["original"]
    .str.split()
    .apply(len)
)

aug_examples["augmented_tokens"] = (
    aug_examples["augmented"]
    .str.split()
    .apply(len)
)

aug_examples["token_diff"] = (
    aug_examples["original_tokens"]
    - aug_examples["augmented_tokens"]
).abs()

display(
    aug_examples
    .sort_values("token_diff", ascending=False)
    .head(20)
)

,original,augmented,original_tokens,augmented_tokens,token_diff
7,muhshulova sanifa rizoevna,muhshulova sanifa,3,2,1
3,abduvalieva barno muhamedchonovna,abduvalieva barno,3,2,1
17,amonov raҳmullo ҳabibovich,amonov raҳmullo,3,2,1
16,azizov ҳasan saidovich,azizov ҳasan,3,2,1
38,dodohanov minavar imomhonovich,dodohanov minavar,3,2,1
32,sattorov dilovar azimchonovich,sattorov dilovar,3,2,1
48,yusupov ҳasan safaarovich,yusupov ҳasan,3,2,1
5,kie muddinov suҳrob amridinovich,kie muddinov suҳrob amridinovich,4,4,0
1,samadov iskandar davlyatovich,samadov iskandar davlyatovich,3,3,0
0,nugmanov dorobiddin meliknorovich,nugmanov dorobiddin eliknorovich,3,3,0


In [5]:
# ============================================================
# REBUILD AUGMENTED TRAIN V2
# ============================================================

from src.features.augmentation import augment_train_pairs

train_augmented_v2 = augment_train_pairs(
    train_pairs=train_pairs,
    n_positive_aug=30_000,
    n_hard_negative_aug=30_000,
    random_state=42,
)

print(f"Rows before: {len(train_pairs):,}")
print(f"Rows after:  {len(train_augmented_v2):,}")

print(train_augmented_v2["label"].value_counts())

print("\nPair types:")
print(train_augmented_v2["pair_type"].value_counts())

Rows before: 120,894
Rows after:  180,894
label
0    115619
1     65275
Name: count, dtype: int64

Pair types:
pair_type
hard_negative_similar_name           50062
easy_negative_different_public_id    35557
positive_same_public_id              35275
augmented_hard_negative              30000
augmented_positive                   30000
Name: count, dtype: int64


In [6]:
# ============================================================
# SAVE AUGMENTED TRAIN V2
# ============================================================

save_path = "../data/processed/train_pairs_augmented_v2.parquet"

train_augmented_v2.to_parquet(
    save_path,
    index=False,
)

print(f"Saved to: {save_path}")

Saved to: ../data/processed/train_pairs_augmented_v2.parquet
